# Finding all HST UV supernova spectra

the whole point of the project: a real query for every SN with HST UV spectroscopy, not relying on paper/abstract searches or the proposer's target name (which is wildly inconsistent - SN2020fqv was logged as `TESS-SN`).

the trick: MAST's HST obs table has a `target_classification` field (the proposer's Phase II class), with values like `EXT-STAR;SUPERNOVA TYPE IA`. filtering on `*supernova*` catches SNe regardless of what they named the target. then we dedup by coordinates (same SN gets logged under SN2023IXF / SN-2023IXF / SN2023IXF-COS etc) to get unique SNe.


In [1]:
import numpy as np
from collections import Counter
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.mast import Observations

# all STIS + COS spectroscopy classified as a supernova
insts = ['STIS/CCD', 'STIS/NUV-MAMA', 'STIS/FUV-MAMA', 'COS/FUV', 'COS/NUV']
sn = Observations.query_criteria(obs_collection='HST', dataproduct_type='spectrum',
                                 target_classification='*upernova*', instrument_name=insts)
print(f"{len(sn)} SN-classified STIS/COS spectra")

# split the transients from the old remnants (SNR). project cares about SN transients
sn['is_remnant'] = np.array(['SNR' in str(c) or 'REMNANT' in str(c) for c in sn['target_classification']])
print(f"  {(~sn['is_remnant']).sum()} transient-SN rows, {sn['is_remnant'].sum()} remnant/SNR rows")


1591 SN-classified STIS/COS spectra
  1504 transient-SN rows, 87 remnant/SNR rows


In [2]:
# dedup by coordinates: same SN gets logged under many name spellings, so group obs within 5 arcsec
c = SkyCoord(sn['s_ra'] * u.deg, sn['s_dec'] * u.deg)
grp = -np.ones(len(c), dtype=int)
g = 0
for i in range(len(c)):
    if grp[i] >= 0:
        continue
    near = (c[i].separation(c) < 5 * u.arcsec) & (grp < 0)
    grp[near] = g
    g += 1
sn['sn_id'] = grp
print(f"{g} unique SNe (5 arcsec grouping) from {len(sn)} spectra")


140 unique SNe (5 arcsec grouping) from 1591 spectra


In [3]:
# build one row per unique SN: representative name, n spectra, coords, instruments, gratings, type
uv_grats = {'G140L', 'G140M', 'G230L', 'G230LB', 'G230M', 'G230MB',
            'G130M', 'G140L', 'G160M', 'G185M', 'G225M', 'G285M', 'G230L'}
rows = []
for gid in range(g):
    sub = sn[sn['sn_id'] == gid]
    nm = Counter(str(x) for x in sub['target_name']).most_common(1)[0][0]
    grats = sorted(set(str(x) for x in sub['filters']))
    cls = Counter(str(x) for x in sub['target_classification']).most_common(1)[0][0]
    has_uv = any(any(u in gr for u in uv_grats) for gr in grats)
    rows.append((nm, len(sub), round(float(np.median(sub['s_ra'])), 5), round(float(np.median(sub['s_dec'])), 5),
                 ';'.join(sorted(set(str(x).split('/')[0] for x in sub['instrument_name']))),
                 ';'.join(grats), bool(sub['is_remnant'].all()), has_uv, cls))

cat = Table(rows=rows, names=['name', 'n_spec', 'ra', 'dec', 'instr', 'gratings', 'is_remnant', 'has_uv', 'classification'])
cat.sort('n_spec')
cat.reverse()
print(f"{len(cat)} unique SNe | {(~cat['is_remnant']).sum()} transients, {cat['is_remnant'].sum()} remnants | {cat['has_uv'].sum()} have a UV grating")
cat.write('../output/uv_sn_catalog.csv', overwrite=True)
cat['name', 'n_spec', 'instr', 'gratings', 'is_remnant', 'classification'][:25]


C:\Users\eluru\AppData\Roaming\Python\Python314\site-packages\numpy\_core\fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedColumn.
  a.partition(kth, axis=axis, kind=kind, order=order)


140 unique SNe | 139 transients, 1 remnants | 102 have a UV grating


name,n_spec,instr,gratings,is_remnant,classification
str18,int64,str8,str42,bool,str79
LMC-SN1987A-STIS-2,203,STIS,G140L;G140M;G230L;G430L;G430M;G750L;G750M,False,EXT-STAR;SUPERNOVA TYPE II
SN2023IXF,133,COS;STIS,G140L;G230L;G230LB;G430L;G750L;G750M,False,EXT-STAR;SUPERNOVA TYPE II
SN2010JL,72,COS;STIS,G130M;G140L;G160M;G230L;G230LB;G430L,False,STAR;SUPERNOVA;SUPERNOVA TYPE II
PTF11KLY,58,STIS,G140L;G230L;G230LB;G430L;G750L,False,EXT-STAR;SUPERNOVA TYPE IA
M82-SN,45,STIS,G230L;G430L;G750L,False,STAR;SUPERNOVA TYPE IA
PTF12GZK,36,STIS,G230L;G430L;G750L,False,STAR;SUPERNOVA
SN1998S,34,STIS,E230M;G140L;G230L;G430L,False,EXT-STAR;SUPERNOVA TYPE II
SN-2005IP,32,STIS,G140L;G230L,False,EXT-STAR;SUPERNOVA;SUPERNOVA TYPE II
ASASSN14LP,32,STIS,G230L;G430L;G750L,False,EXT-STAR;SUPERNOVA TYPE IA


In [5]:
# sanity check: our test SNe should all be in the catalog, including SN2020fqv that's logged as TESS-SN
fqv = SkyCoord(189.1386 * u.deg, 11.2317 * u.deg)   # SN2020fqv true coords (in NGC 4568)
catc = SkyCoord(cat['ra'] * u.deg, cat['dec'] * u.deg)
i = int(catc.separation(fqv).argmin())
print("SN2020fqv ->", cat['name'][i], "| sep", round(catc[i].separation(fqv).arcsec, 2), "arcsec |", cat['gratings'][i])

for probe in ['2024ISS', '2023IXF', '2010JL']:
    hit = [r['name'] for r in cat if probe in str(r['name']).upper()]
    print(f"{probe}: {hit}")


SN2020fqv -> TESS-SN | sep 0.08 arcsec | G130M;G160M;G230L;G230LB;G430L
2024ISS: [np.str_('SN2024ISS')]
2023IXF: [np.str_('SN2023IXF')]
2010JL: [np.str_('SN2010JL')]
